In [ ]:
# =========================
# CELL 1: setup + install
# =========================

import os
import random
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

# --- Environment detection ---
try:
    from google.colab import drive

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/moses_full_reproduction")
    REPO_DIR = Path("/content/MoSEs")
    REPO_CACHE = DRIVE_ROOT / "repo_cache"
    HF_CACHE = DRIVE_ROOT / "hf_cache"
    RESULTS_ROOT = DRIVE_ROOT / "final_outputs"
else:
    # Local: use the repo directory directly
    REPO_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
    # If running from inside the repo, use the repo root
    if (REPO_DIR / "requirements.txt").exists():
        pass
    elif (REPO_DIR.parent / "requirements.txt").exists():
        REPO_DIR = REPO_DIR.parent
    HF_CACHE = REPO_DIR / ".cache" / "huggingface"
    RESULTS_ROOT = REPO_DIR / "outputs"
    DRIVE_ROOT = None
    REPO_CACHE = None

dirs_to_create = [HF_CACHE, RESULTS_ROOT]
if IN_COLAB:
    dirs_to_create += [DRIVE_ROOT, REPO_CACHE]
for p in dirs_to_create:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def run_cmd(cmd, cwd=None, env=None, shell=False, check=True):
    printable = (
        cmd if isinstance(cmd, str) else " ".join(shlex.quote(str(x)) for x in cmd)
    )
    print("\n" + "=" * 100)
    print(printable)
    print("=" * 100)
    result = subprocess.run(
        cmd, cwd=cwd, env=env, shell=shell, capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode,
            cmd,
            output=result.stdout,
            stderr=result.stderr,
        )
    return result


def copytree_merge(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        s = item
        d = dst / item.name
        if s.is_dir():
            copytree_merge(s, d)
        else:
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(s, d)


if IN_COLAB:
    # Only clone if repo doesn't exist; pull latest otherwise
    if not REPO_DIR.exists():
        run_cmd(
            ["git", "clone", "https://github.com/HuskyDevClub/MoSEs.git", str(REPO_DIR)]
        )
    else:
        print(f"[SKIP] Repo already exists at {REPO_DIR}, pulling latest...")
        run_cmd(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=False)

    # Restore cached outputs only if not already present locally
    for rel in ["weights", "logs"]:
        local_dir = REPO_DIR / rel
        if not local_dir.exists() or not list(local_dir.glob("*")):
            copytree_merge(REPO_CACHE / rel, local_dir)
        else:
            print(f"[SKIP] {rel}/ already exists locally, skipping Drive restore.")

    cached_data_dir = REPO_CACHE / "data"
    if cached_data_dir.exists():
        for p in cached_data_dir.glob("split_datasets*"):
            local_split = REPO_DIR / "data" / p.name
            if not local_split.exists() or not list(local_split.glob("*.json")):
                copytree_merge(p, local_split)
            else:
                print(
                    f"[SKIP] {p.name} already exists locally, skipping Drive restore."
                )

# install dependencies
run_cmd(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
)

run_cmd(["nvidia-smi"], check=False)
print("Setup complete.")

In [ ]:
# =========================
# CELL 2: configuration + helpers
# =========================

import json
import re
import pandas as pd
from pathlib import Path

DATA_INPUT_DIR = REPO_DIR / "data" / "doc4split"

MAIN_KEYS = ["cmv", "sci", "wp", "xsum"]
LOW_KEYS = ["cnn", "dialogsum", "imdb", "pubmedqa"]

# You can flip any of these to False if you need to rerun only part of the pipeline.
RUN_FAST = True
RUN_LASTDE = True
RUN_ROBERTA = True

# Official example scripts clearly show main fast/lastde use *_train as reference set.
# The released roberta example points to the unsplit folder; keep this toggle if you want to match that behavior.
ROBERTA_MAIN_USE_FULL_FOLDER = True

DETECTORS = {
    "fast": {
        "enabled": RUN_FAST,
        "crit_type": "fast",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_fast",
        "extra_args": [],
    },
    "lastde": {
        "enabled": RUN_LASTDE,
        "crit_type": "lastde",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_lastde",
        "extra_args": ["--embed_size", "4", "--epsilon", "8", "--tau_prime", "15"],
    },
    "roberta_base": {
        "enabled": RUN_ROBERTA,
        "crit_type": "roberta",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_main_1000_roberta_base",
        "extra_args": ["--roberta_model_name", "roberta-base-openai-detector"],
    },
}


def sync_outputs_to_drive():
    if not IN_COLAB:
        return
    (REPO_CACHE / "data").mkdir(parents=True, exist_ok=True)
    for p in (REPO_DIR / "data").glob("split_datasets*"):
        copytree_merge(p, REPO_CACHE / "data" / p.name)
    copytree_merge(REPO_DIR / "weights", REPO_CACHE / "weights")
    copytree_merge(REPO_DIR / "logs", REPO_CACHE / "logs")


def dataset_key_from_json_name(name):
    key = Path(name).stem.lower()
    if key.endswith("_dataset"):
        key = key[:-8]
    return key


def subset_json_folder(src_dir, dst_dir, allowed_keys):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_dir.glob("*.json"):
        if dataset_key_from_json_name(p.name) in allowed_keys:
            shutil.copy2(p, dst_dir / p.name)


def split_json_folder_dataset_aware(
    src_dir, train_dir, test_dir, main_test_per_label=100, seed=42
):
    """Split each JSON file in src_dir into train/test sets.

    Paper specifies 1800 ref / 200 test for main datasets (100 per label)
    and 200 ref / 200 test for low-resource datasets (half per label).
    """
    src_dir = Path(src_dir)
    train_dir = Path(train_dir)
    test_dir = Path(test_dir)
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    for p in sorted(src_dir.glob("*.json")):
        key = dataset_key_from_json_name(p.name)
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)

        label0 = [x for x in data if int(x["label"]) == 0]
        label1 = [x for x in data if int(x["label"]) == 1]
        rng.shuffle(label0)
        rng.shuffle(label1)

        if key in LOW_KEYS:
            # Paper: 200 ref / 200 test  ->  half per label to test, half to reference
            test_n = min(len(label0), len(label1)) // 2
        else:
            # Paper: 1800 ref / 200 test  ->  100 per label for test
            test_n = min(main_test_per_label, len(label0) // 2, len(label1) // 2)

        test_data = label0[:test_n] + label1[:test_n]
        train_data = label0[test_n:] + label1[test_n:]

        with open(test_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(test_data, f, ensure_ascii=False, indent=2)
        with open(train_dir / p.name, "w", encoding="utf-8") as f:
            json.dump(train_data, f, ensure_ascii=False, indent=2)

        print(
            f"{p.name}: train={len(train_data)}, test={len(test_data)}, test_per_label={test_n}"
        )


def prepare_detector_views(processed_dir):
    """Split the full 8-dataset processed folder into train/test, then create
    main-only and low-only test subsets for evaluation iteration.

    Matches the paper: SAR and CTE both operate on the full 8-class SRR,
    while evaluation iterates over 4 main or 4 low-resource test files.
    """
    processed_dir = Path(processed_dir)

    # Full train/test split (all 8 datasets)
    full_train = Path(str(processed_dir) + "_train")
    full_test = Path(str(processed_dir) + "_test")

    # Test subsets for evaluation iteration only
    main_test = Path(str(processed_dir) + "_main_test")
    low_test = Path(str(processed_dir) + "_low_test")

    # Split full folder into train/test (100 per label for main, half for low)
    if (
        not full_train.exists()
        or not list(full_train.glob("*.json"))
        or not full_test.exists()
        or not list(full_test.glob("*.json"))
    ):
        split_json_folder_dataset_aware(
            processed_dir, full_train, full_test, main_test_per_label=100, seed=42
        )

    # Subset the test folder into main / low for CTE evaluation iteration
    if not main_test.exists() or not list(main_test.glob("*.json")):
        subset_json_folder(full_test, main_test, MAIN_KEYS)
    if not low_test.exists() or not list(low_test.glob("*.json")):
        subset_json_folder(full_test, low_test, LOW_KEYS)

    return {
        "full": processed_dir,  # unsplit, all 8 datasets
        "train": full_train,  # reference set, all 8 datasets
        "test": full_test,  # test set, all 8 datasets
        "main_test": main_test,  # 4 main dataset test files
        "low_test": low_test,  # 4 low-resource dataset test files
    }


def find_best_sar_weights(output_dir):
    """Find the SAR weight file with the highest epoch number (= best val accuracy).

    SAR.py saves a new file each time val_acc improves, so the highest
    epoch number corresponds to the best model.
    """
    output_dir = Path(output_dir)
    pts = list(output_dir.glob("subcentroids_head_epoch*.pt"))
    if not pts:
        return None

    def epoch_num(p):
        m = re.search(r"epoch(\d+)", p.name)
        return int(m.group(1)) if m else 0

    return max(pts, key=epoch_num)


def flatten_json(x, prefix=""):
    out = {}
    if isinstance(x, dict):
        for k, v in x.items():
            out.update(flatten_json(v, f"{prefix}{k}." if prefix else f"{k}."))
    elif isinstance(x, list):
        if len(x) <= 20:
            out[prefix[:-1]] = str(x)
    else:
        out[prefix[:-1]] = x
    return out


print("Helpers ready.")

In [ ]:
# =========================
# CELL 2B: metrics computation helpers
# =========================

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


def compute_cte_metrics(results_base_dir, dataset_keys):
    """Compute accuracy, F1, and AUROC from CTE output JSONs.

    Args:
        results_base_dir: Directory containing detector subdirs with *_results.json files.
        dataset_keys: List of dataset keys to include (e.g., MAIN_KEYS or LOW_KEYS).

    Returns:
        DataFrame with columns: detector, dataset, method, accuracy, f1, auroc
    """
    results_base_dir = Path(results_base_dir)
    records = []

    for det_dir in sorted(results_base_dir.iterdir()):
        if not det_dir.is_dir():
            continue
        det_name = det_dir.name

        for json_file in sorted(det_dir.glob("*_results.json")):
            ds_key = dataset_key_from_json_name(json_file.name.replace("_results", ""))
            if ds_key not in dataset_keys:
                continue

            with open(json_file, "r") as f:
                data = json.load(f)

            for method_name, method_data in data.items():
                y_true = method_data["y_true"]
                y_pred = method_data["y_pred"]
                ai_prob = method_data.get("ai_prob", [])

                acc = accuracy_score(y_true, y_pred)
                f1_val = f1_score(y_true, y_pred, average="binary")

                auroc = None
                try:
                    if ai_prob and len(set(ai_prob)) > 1:
                        auroc = roc_auc_score(y_true, ai_prob)
                except Exception:
                    pass

                records.append(
                    {
                        "detector": det_name,
                        "dataset": ds_key,
                        "method": method_name,
                        "accuracy": acc,
                        "f1": f1_val,
                        "auroc": auroc,
                    }
                )

    return pd.DataFrame(records)


# Paper method names for display
METHOD_LABELS = {
    "full_constant": "Baseline-Const",
    "full_count": "Baseline-Count",
    "constant": "SAR-Const",
    "count": "SAR-Count",
    "logistic": "MoSEs-lr",
    "xg": "MoSEs-xg",
}
METHOD_ORDER = list(METHOD_LABELS.keys())


def display_results_table(df, metric, title):
    """Pivot metrics DataFrame into a paper-style comparison table."""
    rows = []
    for det in sorted(df["detector"].unique()):
        for method in METHOD_ORDER:
            subset = df[(df["detector"] == det) & (df["method"] == method)]
            if subset.empty:
                continue
            row = {"Detector": det, "Method": METHOD_LABELS[method]}
            for _, r in subset.iterrows():
                row[r["dataset"].upper()] = r[metric]
            numeric_vals = [
                v for v in row.values() if isinstance(v, (int, float)) and pd.notna(v)
            ]
            row["Avg"] = sum(numeric_vals) / len(numeric_vals) if numeric_vals else None
            rows.append(row)

    table = pd.DataFrame(rows)
    print(f"\n{'='*80}")
    print(f"  {title} ({metric})")
    print(f"{'='*80}")
    print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return table


def compute_improvement(
    df,
    moses_methods=("logistic", "xg"),
    baseline_methods=("full_constant", "full_count"),
):
    """Compute average improvement of MoSEs methods over baselines."""
    moses_acc = df[df["method"].isin(moses_methods)]["accuracy"].mean()
    baseline_acc = df[df["method"].isin(baseline_methods)]["accuracy"].mean()
    if baseline_acc > 0:
        improvement = ((moses_acc - baseline_acc) / baseline_acc) * 100
    else:
        improvement = 0.0
    return moses_acc, baseline_acc, improvement


print("Metrics helpers ready.")

In [ ]:
# =========================
# CELL 3: verify input data + build processed folders for all detectors
# =========================

csvs = sorted(DATA_INPUT_DIR.glob("*.csv"))
print("Found CSV files:")
for p in csvs:
    print(" -", p.name)

assert len(csvs) > 0, f"No CSV files found in {DATA_INPUT_DIR}"

# Count the actual number of unique datasets in the input CSVs
# so the skip threshold matches reality (e.g. 4 for main-only, 8 for full)
import csv as _csv

_expected_datasets = set()
for _csv_path in csvs:
    with open(_csv_path, "r", encoding="utf-8") as _f:
        reader = _csv.DictReader(_f)
        for row in reader:
            src = row.get("src", "")
            idx = src.find("_")
            name = src[:idx].lower() if idx != -1 else src.lower()
            if name:
                _expected_datasets.add(name)
EXPECTED_NUM_DATASETS = len(_expected_datasets)
print(f"Expected datasets ({EXPECTED_NUM_DATASETS}): {sorted(_expected_datasets)}")


def build_processed_folder(det_name, cfg):
    out_dir = REPO_DIR / "data" / cfg["out_name"]
    jsons = list(out_dir.glob("*.json")) if out_dir.exists() else []
    if len(jsons) >= EXPECTED_NUM_DATASETS:
        print(f"[SKIP] {det_name}: found {len(jsons)} processed JSONs in {out_dir}")
        return out_dir

    cmd = [
        sys.executable,
        "split_datasets.py",
        "--input_directory",
        str(DATA_INPUT_DIR),
        "--output_directory",
        str(out_dir),
        "--embedding_type",
        "encode",
        "--embedding_model_name",
        "BAAI/bge-m3",
        "--scoring_model_name",
        "EleutherAI/gpt-neo-2.7B",
        "--batch_size",
        "500",
        "--crit_type",
        cfg["crit_type"],
        "--cache_dir",
        str(HF_CACHE),
    ] + cfg["extra_args"]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()
    return out_dir


PROCESSED_DIRS = {}

for det_name, cfg in DETECTORS.items():
    if not cfg["enabled"]:
        continue
    PROCESSED_DIRS[det_name] = build_processed_folder(det_name, cfg)

print("\nProcessed dirs:")
for k, v in PROCESSED_DIRS.items():
    print(k, "->", v)

In [ ]:
# =========================
# CELL 4: create train/test splits and test subsets for all detectors
# =========================

VIEW_PATHS = {}

for det_name, processed_dir in PROCESSED_DIRS.items():
    print(f"\nPreparing views for {det_name}")
    VIEW_PATHS[det_name] = prepare_detector_views(processed_dir)

sync_outputs_to_drive()

for det_name, views in VIEW_PATHS.items():
    print(f"\n=== {det_name} ===")
    for k, v in views.items():
        count = len(list(Path(v).glob("*.json"))) if Path(v).exists() else 0
        print(f"{k:>12}: {v} ({count} json files)")

In [ ]:
# =========================
# CELL 5: train SAR on the full 8-class SRR (matching the paper)
# =========================

# Paper Section 3.3: SAR operates on the full SRR containing all 8 stylistic categories.
# The original run_train.sh trains SAR on the complete (unsplit) fast-processed folder.
# We train on the _train split (all 8 datasets) so test data is excluded.

SAR_OUT = REPO_DIR / "weights" / "model_output_c16"


def train_sar_if_needed(dataset_folder, output_dir):
    class_path = output_dir / "class_names.json"
    weight_path = find_best_sar_weights(output_dir)

    if weight_path is not None and class_path.exists():
        print(f"[SKIP] SAR already exists at {output_dir} (best: {weight_path.name})")
        return weight_path, class_path

    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "SAR.py",
        "train",
        "--model_name",
        "BAAI/bge-m3",
        "--datasets_folder",
        str(dataset_folder),
        "--embedding_type",
        "encode",
        "--num_epochs",
        "100",
        "--batch_size",
        "32",
        "--num_subcentroids",
        "4",
        "--output_dir",
        str(output_dir),
    ]

    # If a checkpoint exists from a previous interrupted run, resume from it
    checkpoint_path = output_dir / "checkpoint_latest.pt"
    if checkpoint_path.exists():
        print(f"[RESUME] Found checkpoint at {checkpoint_path}, resuming training...")
        cmd += ["--resume_checkpoint", str(checkpoint_path)]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()

    weight_path = find_best_sar_weights(output_dir)
    assert (
        weight_path is not None
    ), f"No SAR weights found in {output_dir} after training"
    return weight_path, class_path


assert (
    "fast" in VIEW_PATHS
), "fast detector processed folder is needed to train SAR as in the released examples."

# Train a single SAR on the full _train folder (all 8 datasets)
SAR_PATH, SAR_CLASS = train_sar_if_needed(VIEW_PATHS["fast"]["train"], SAR_OUT)

print("SAR weights:", SAR_PATH)
print("SAR classes:", SAR_CLASS)

In [ ]:
# =========================
# CELL 6: run CTE evaluation for all detectors on main + low-resource
# =========================

# Paper: CTE uses the full SRR (all 8 datasets) as the reference pool.
# Original run_cte_fast_detect.sh uses the _train folder (all 8 datasets).
# Original run_cte_roberta_base.sh uses the unsplit folder (all 8 datasets).

from concurrent.futures import ThreadPoolExecutor, as_completed

DONE_DIR = REPO_DIR / "logs" / "done_markers"
DONE_DIR.mkdir(parents=True, exist_ok=True)

# Per-sample work (LR + XGBoost training) is CPU-bound, not GPU-bound.
# GPU is only used briefly for model load + routing, so we can safely run
# more workers than GPU memory alone would suggest.
# A good default: number of CPU cores (or half if you want headroom).
MAX_CTE_WORKERS = 4


def run_cte_single(
    test_file, result_prefix, datasets_folder, sar_path, class_path, marker_prefix
):
    """Run CTE.py on a single test file. Returns (marker_name, success, error)."""
    test_file = Path(test_file)
    result_prefix = Path(result_prefix)
    result_prefix.parent.mkdir(parents=True, exist_ok=True)
    marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"

    if marker.exists():
        return marker.name, True, "skipped"

    cmd = [
        sys.executable,
        "CTE.py",
        "--file_path",
        str(test_file),
        "--result_file_path",
        str(result_prefix),
        "--datasets_folder",
        str(datasets_folder),
        "--sar_path",
        str(sar_path),
        "--class_path",
        str(class_path),
    ]
    try:
        result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        if result.returncode != 0:
            return marker.name, False, result.stderr
        marker.touch()
        return marker.name, True, result.stdout[-200:] if result.stdout else ""
    except Exception as e:
        return marker.name, False, str(e)


# Collect all CTE jobs
cte_jobs = []

for det_name, views in VIEW_PATHS.items():
    if det_name == "roberta_base" and ROBERTA_MAIN_USE_FULL_FOLDER:
        ref_folder = views["full"]
    else:
        ref_folder = views["train"]

    # Main test files
    main_test_dir = Path(views["main_test"])
    for test_file in sorted(main_test_dir.glob("*.json")):
        cte_jobs.append(
            {
                "test_file": test_file,
                "result_prefix": REPO_DIR / "logs" / "reproduction" / "main" / det_name,
                "datasets_folder": ref_folder,
                "sar_path": SAR_PATH,
                "class_path": SAR_CLASS,
                "marker_prefix": f"main__{det_name}",
                "label": f"{det_name}/main/{test_file.stem}",
            }
        )

    # Low-resource test files
    low_test_dir = Path(views["low_test"])
    for test_file in sorted(low_test_dir.glob("*.json")):
        cte_jobs.append(
            {
                "test_file": test_file,
                "result_prefix": REPO_DIR / "logs" / "reproduction" / "low" / det_name,
                "datasets_folder": views["train"],
                "sar_path": SAR_PATH,
                "class_path": SAR_CLASS,
                "marker_prefix": f"low__{det_name}",
                "label": f"{det_name}/low/{test_file.stem}",
            }
        )

print(f"Total CTE jobs: {len(cte_jobs)}, max parallel workers: {MAX_CTE_WORKERS}")

# Run jobs in parallel
completed = 0
failed = []

with ThreadPoolExecutor(max_workers=MAX_CTE_WORKERS) as executor:
    future_to_label = {}
    for job in cte_jobs:
        future = executor.submit(
            run_cte_single,
            job["test_file"],
            job["result_prefix"],
            job["datasets_folder"],
            job["sar_path"],
            job["class_path"],
            job["marker_prefix"],
        )
        future_to_label[future] = job["label"]

    for future in as_completed(future_to_label):
        label = future_to_label[future]
        marker_name, success, msg = future.result()
        completed += 1
        status = "OK" if success else "FAIL"
        print(f"[{completed}/{len(cte_jobs)}] {status}: {label} ({msg[:80]})")
        if not success:
            failed.append((label, msg))

sync_outputs_to_drive()

if failed:
    print(f"\n{len(failed)} jobs FAILED:")
    for label, msg in failed:
        print(f"  {label}: {msg[:200]}")
else:
    print(f"\nAll {len(cte_jobs)} CTE jobs finished successfully.")

In [ ]:
# =========================
# CELL 7: compute and display main results (Table 1)
# =========================

main_results_dir = REPO_DIR / "logs" / "reproduction" / "main"
main_metrics_df = compute_cte_metrics(main_results_dir, MAIN_KEYS)

if main_metrics_df.empty:
    print("No main results found. Run Cell 6 (CTE evaluation) first.")
else:
    # Table 1: Detection performance
    acc_table = display_results_table(
        main_metrics_df, "accuracy", "Table 1: Main Results"
    )
    f1_table = display_results_table(main_metrics_df, "f1", "Table 1: Main Results")

    # AUROC (only for methods with probability outputs)
    auroc_df = main_metrics_df[main_metrics_df["auroc"].notna()]
    if not auroc_df.empty:
        auroc_table = display_results_table(auroc_df, "auroc", "Table 1: Main Results")

    # H1: Average improvement over baselines
    moses_acc, baseline_acc, improvement = compute_improvement(main_metrics_df)
    print(f"\n{'='*80}")
    print(f"  H1 Verification: MoSEs vs Baselines (Accuracy)")
    print(f"{'='*80}")
    print(f"  MoSEs avg accuracy:    {moses_acc:.4f}")
    print(f"  Baseline avg accuracy: {baseline_acc:.4f}")
    print(f"  Relative improvement:  {improvement:+.2f}%")
    print(f"  Paper claims:          +11.34%")

    # Save detailed metrics CSV
    main_metrics_csv = RESULTS_ROOT / "main_metrics_detailed.csv"
    main_metrics_df.to_csv(main_metrics_csv, index=False)
    print(f"\nDetailed metrics saved to: {main_metrics_csv}")

# Archive logs and weights
logs_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_logs"), "zip", REPO_DIR / "logs"
)
weights_zip = shutil.make_archive(
    str(RESULTS_ROOT / "moses_weights"), "zip", REPO_DIR / "weights"
)
print(f"\nLogs zip   : {logs_zip}")
print(f"Weights zip: {weights_zip}")

sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 8A: stage + validate low-resource raw CSVs
# =========================

import pandas as pd
import shutil
from pathlib import Path

LOW_INPUT_DIR = REPO_DIR / "data" / "doc4split_low"
LOW_INPUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    LOW_RAW_SRC_DIR = DRIVE_ROOT / "low_resource_csvs"
    # Clear staged low input dir, then copy fresh CSVs from Drive
    for p in LOW_INPUT_DIR.glob("*.csv"):
        p.unlink()
    src_csvs = sorted(LOW_RAW_SRC_DIR.glob("*.csv"))
    assert src_csvs, (
        f"No low-resource CSV files found in {LOW_RAW_SRC_DIR}.\n"
        "Place your low-resource raw CSV(s) there first."
    )
    for p in src_csvs:
        shutil.copy2(p, LOW_INPUT_DIR / p.name)
else:
    # Local: expect CSVs already placed in doc4split_low
    existing = sorted(LOW_INPUT_DIR.glob("*.csv"))
    if not existing:
        print(
            f"WARNING: No CSV files found in {LOW_INPUT_DIR}.\n"
            f"Place your low-resource raw CSV(s) there before running this cell."
        )

print("Staged low-resource CSV files:")
for p in sorted(LOW_INPUT_DIR.glob("*.csv")):
    print(" -", p.name)

# Validate that the staged CSVs actually contain the 4 low-resource dataset prefixes
required_low = set(LOW_KEYS)
seen_low = set()

for csv_path in sorted(LOW_INPUT_DIR.glob("*.csv")):
    df = pd.read_csv(csv_path)
    missing_cols = {"src", "text", "label"} - set(df.columns)
    assert not missing_cols, f"{csv_path.name} is missing columns: {missing_cols}"

    prefixes = (
        df["src"]
        .dropna()
        .astype(str)
        .map(lambda s: s.split("_", 1)[0].lower())
        .unique()
        .tolist()
    )
    seen_low.update(prefixes)

print("\nDetected low-resource src prefixes:", sorted(seen_low))
missing_low = required_low - seen_low
assert not missing_low, (
    f"Missing low-resource dataset prefixes in staged CSVs: {sorted(missing_low)}\n"
    f"Need all of: {sorted(required_low)}"
)

print("\nLow-resource raw data is staged and validated.")

In [ ]:
# =========================
# CELL 8B: build processed low-resource folders for all detectors
# =========================

LOW_DETECTORS = {
    "fast": {
        "enabled": RUN_FAST,
        "crit_type": "fast",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_low_200_fast",
        "extra_args": [],
    },
    "lastde": {
        "enabled": RUN_LASTDE,
        "crit_type": "lastde",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_low_200_lastde",
        "extra_args": ["--embed_size", "4", "--epsilon", "8", "--tau_prime", "15"],
    },
    "roberta_base": {
        "enabled": RUN_ROBERTA,
        "crit_type": "roberta",
        "out_name": "split_datasets_bge_m3_tiny_encode_crit_cond6_low_200_roberta_base",
        "extra_args": ["--roberta_model_name", "roberta-base-openai-detector"],
    },
}


def has_expected_jsons(folder, expected_keys):
    folder = Path(folder)
    if not folder.exists():
        return False
    found = {dataset_key_from_json_name(p.name) for p in folder.glob("*.json")}
    return set(expected_keys).issubset(found)


def build_low_processed_folder(det_name, cfg):
    out_dir = REPO_DIR / "data" / cfg["out_name"]

    if has_expected_jsons(out_dir, LOW_KEYS):
        print(f"[SKIP] {det_name}: found expected low-resource JSONs in {out_dir}")
        return out_dir

    if out_dir.exists():
        shutil.rmtree(out_dir)

    cmd = [
        sys.executable,
        "split_datasets.py",
        "--input_directory",
        str(LOW_INPUT_DIR),
        "--output_directory",
        str(out_dir),
        "--embedding_type",
        "encode",
        "--embedding_model_name",
        "BAAI/bge-m3",
        "--scoring_model_name",
        "EleutherAI/gpt-neo-2.7B",
        "--batch_size",
        "500",
        "--crit_type",
        cfg["crit_type"],
        "--cache_dir",
        str(HF_CACHE),
    ] + cfg["extra_args"]

    run_cmd(cmd, cwd=REPO_DIR)
    sync_outputs_to_drive()

    found = sorted(p.name for p in out_dir.glob("*.json"))
    print(f"\nProcessed low-resource JSONs for {det_name}:")
    for name in found:
        print(" -", name)

    assert has_expected_jsons(
        out_dir, LOW_KEYS
    ), f"{det_name}: expected low-resource JSONs were not all created in {out_dir}"
    return out_dir


LOW_PROCESSED_DIRS = {}

for det_name, cfg in LOW_DETECTORS.items():
    if not cfg["enabled"]:
        continue
    LOW_PROCESSED_DIRS[det_name] = build_low_processed_folder(det_name, cfg)

print("\nLOW_PROCESSED_DIRS:")
for k, v in LOW_PROCESSED_DIRS.items():
    print(k, "->", v)

In [ ]:
# =========================
# CELL 8C: create low-resource train/test splits
# =========================


def prepare_low_detector_views(processed_dir):
    """
    Low-resource-only pipeline:
    - processed_dir contains only cnn/dialogsum/imdb/pubmedqa JSONs
    - split_json_folder_dataset_aware will create 200 ref / 200 test per dataset
      when each JSON has 400 rows balanced by label
    """
    processed_dir = Path(processed_dir)

    full_train = Path(str(processed_dir) + "_train")
    full_test = Path(str(processed_dir) + "_test")
    low_test = Path(str(processed_dir) + "_low_test")

    need_split = (
        not full_train.exists()
        or not full_test.exists()
        or not has_expected_jsons(full_train, LOW_KEYS)
        or not has_expected_jsons(full_test, LOW_KEYS)
    )
    if need_split:
        if full_train.exists():
            shutil.rmtree(full_train)
        if full_test.exists():
            shutil.rmtree(full_test)
        split_json_folder_dataset_aware(
            processed_dir,
            full_train,
            full_test,
            main_test_per_label=100,  # ignored for LOW_KEYS; function uses half-per-label
            seed=42,
        )

    if low_test.exists():
        shutil.rmtree(low_test)
    subset_json_folder(full_test, low_test, LOW_KEYS)

    return {
        "full": processed_dir,
        "train": full_train,
        "test": full_test,
        "low_test": low_test,
    }


LOW_VIEW_PATHS = {}

for det_name, processed_dir in LOW_PROCESSED_DIRS.items():
    print(f"\nPreparing low-resource views for {det_name}")
    LOW_VIEW_PATHS[det_name] = prepare_low_detector_views(processed_dir)

sync_outputs_to_drive()

for det_name, views in LOW_VIEW_PATHS.items():
    print(f"\n=== LOW {det_name} ===")
    for k, v in views.items():
        count = len(list(Path(v).glob("*.json"))) if Path(v).exists() else 0
        print(f"{k:>10}: {v} ({count} json files)")

In [ ]:
# =========================
# CELL 8D: train a separate SAR for the low-resource SRR
# =========================

LOW_SAR_OUT = REPO_DIR / "weights" / "model_output_low_c4"

assert (
    "fast" in LOW_VIEW_PATHS
), "fast low-resource processed folder is needed to train low-resource SAR."

LOW_SAR_PATH, LOW_SAR_CLASS = train_sar_if_needed(
    LOW_VIEW_PATHS["fast"]["train"],
    LOW_SAR_OUT,
)

print("LOW_SAR_PATH :", LOW_SAR_PATH)
print("LOW_SAR_CLASS:", LOW_SAR_CLASS)
sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 8E: run CTE evaluation on low-resource datasets only
# =========================

DONE_DIR = REPO_DIR / "logs" / "done_markers"
DONE_DIR.mkdir(parents=True, exist_ok=True)


def run_cte_folder_safe(
    test_folder, datasets_folder, result_prefix, sar_path, class_path, marker_prefix
):
    test_folder = Path(test_folder)
    result_prefix = Path(result_prefix)
    result_prefix.mkdir(parents=True, exist_ok=True)

    test_files = sorted(test_folder.glob("*.json"))
    assert test_files, f"No test JSON files found in {test_folder}"

    for test_file in test_files:
        marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"
        if marker.exists():
            print(f"[SKIP] {marker.name}")
            continue

        cmd = [
            sys.executable,
            "CTE.py",
            "--file_path",
            str(test_file),
            "--result_file_path",
            str(result_prefix),
            "--datasets_folder",
            str(datasets_folder),
            "--sar_path",
            str(sar_path),
            "--class_path",
            str(class_path),
        ]
        run_cmd(cmd, cwd=REPO_DIR)
        marker.touch()
        sync_outputs_to_drive()


for det_name, views in LOW_VIEW_PATHS.items():
    print(f"\n==================== {det_name}: LOW-RESOURCE ====================")

    # use the clean 200-reference split for all detectors in the low-resource run
    ref_folder = views["train"]

    run_cte_folder_safe(
        test_folder=views["low_test"],
        datasets_folder=ref_folder,
        result_prefix=REPO_DIR / "logs" / "reproduction" / "low_only" / det_name,
        sar_path=LOW_SAR_PATH,
        class_path=LOW_SAR_CLASS,
        marker_prefix=f"low_only__{det_name}",
    )

print("All low-resource CTE runs finished.")
sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 8F: compute and display low-resource results (Table 2)
# =========================

low_results_dir = REPO_DIR / "logs" / "reproduction" / "low_only"
low_metrics_df = compute_cte_metrics(low_results_dir, LOW_KEYS)

if low_metrics_df.empty:
    print("No low-resource results found. Run Cell 8E first.")
else:
    # Table 2: Low-resource detection performance
    acc_table_low = display_results_table(
        low_metrics_df, "accuracy", "Table 2: Low-Resource Results"
    )
    f1_table_low = display_results_table(
        low_metrics_df, "f1", "Table 2: Low-Resource Results"
    )

    # AUROC (only for methods with probability outputs)
    auroc_low = low_metrics_df[low_metrics_df["auroc"].notna()]
    if not auroc_low.empty:
        auroc_table_low = display_results_table(
            auroc_low, "auroc", "Table 2: Low-Resource Results"
        )

    # H2: Average improvement over baselines (low-resource)
    moses_acc, baseline_acc, improvement = compute_improvement(low_metrics_df)
    print(f"\n{'='*80}")
    print(f"  H2 Verification: MoSEs vs Baselines - Low-Resource (Accuracy)")
    print(f"{'='*80}")
    print(f"  MoSEs avg accuracy:    {moses_acc:.4f}")
    print(f"  Baseline avg accuracy: {baseline_acc:.4f}")
    print(f"  Relative improvement:  {improvement:+.2f}%")
    print(f"  Paper claims:          +39.15%")

    # Save detailed metrics CSV
    low_metrics_csv = RESULTS_ROOT / "low_resource_metrics_detailed.csv"
    low_metrics_df.to_csv(low_metrics_csv, index=False)
    print(f"\nDetailed metrics saved to: {low_metrics_csv}")

sync_outputs_to_drive()

In [ ]:
# =========================
# CELL 9A: run CTE ablation experiments (Tables 4 & 5)
# =========================

# Table 4: SAR ablation (--no_router)
# Table 5: Conditional feature ablation (--no_embedding, --no_condition, --pca_dim -1)

ABLATIONS = {
    "no_router": ["--no_router"],
    "no_embedding": ["--no_embedding"],
    "no_condition": ["--no_condition"],
    "no_pca": ["--pca_dim", "-1"],
}

ablation_jobs = []

for abl_name, abl_flags in ABLATIONS.items():
    for det_name, views in VIEW_PATHS.items():
        if det_name == "roberta_base" and ROBERTA_MAIN_USE_FULL_FOLDER:
            ref_folder = views["full"]
        else:
            ref_folder = views["train"]

        main_test_dir = Path(views["main_test"])
        for test_file in sorted(main_test_dir.glob("*.json")):
            marker_prefix = f"abl_{abl_name}__{det_name}"
            marker = DONE_DIR / f"{marker_prefix}__{test_file.stem}.done"

            if marker.exists():
                continue

            ablation_jobs.append(
                {
                    "test_file": test_file,
                    "result_dir": REPO_DIR
                    / "logs"
                    / "reproduction"
                    / "ablation"
                    / abl_name
                    / det_name,
                    "datasets_folder": ref_folder,
                    "sar_path": SAR_PATH,
                    "class_path": SAR_CLASS,
                    "marker_prefix": marker_prefix,
                    "abl_flags": abl_flags,
                    "label": f"{abl_name}/{det_name}/{test_file.stem}",
                }
            )

print(f"Ablation CTE jobs to run: {len(ablation_jobs)} (already-done jobs skipped)")

if ablation_jobs:

    def run_ablation_cte(job):
        test_file = Path(job["test_file"])
        result_dir = Path(job["result_dir"])
        result_dir.mkdir(parents=True, exist_ok=True)
        marker = DONE_DIR / f"{job['marker_prefix']}__{test_file.stem}.done"

        cmd = [
            sys.executable,
            "CTE.py",
            "--file_path",
            str(test_file),
            "--result_file_path",
            str(result_dir),
            "--datasets_folder",
            str(job["datasets_folder"]),
            "--sar_path",
            str(job["sar_path"]),
            "--class_path",
            str(job["class_path"]),
        ] + job["abl_flags"]

        try:
            result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
            if result.returncode != 0:
                return job["label"], False, result.stderr[:200]
            marker.touch()
            return job["label"], True, "ok"
        except Exception as e:
            return job["label"], False, str(e)

    completed = 0
    failed = []

    with ThreadPoolExecutor(max_workers=MAX_CTE_WORKERS) as executor:
        future_to_label = {
            executor.submit(run_ablation_cte, job): job["label"]
            for job in ablation_jobs
        }
        for future in as_completed(future_to_label):
            label = future_to_label[future]
            lbl, success, msg = future.result()
            completed += 1
            status = "OK" if success else "FAIL"
            print(f"[{completed}/{len(ablation_jobs)}] {status}: {lbl} ({msg[:60]})")
            if not success:
                failed.append((lbl, msg))

    sync_outputs_to_drive()

    if failed:
        print(f"\n{len(failed)} ablation jobs FAILED:")
        for lbl, msg in failed:
            print(f"  {lbl}: {msg[:200]}")
    else:
        print(f"\nAll {len(ablation_jobs)} ablation jobs finished.")
else:
    print("All ablation jobs already done (skipped).")

In [ ]:
# =========================
# CELL 9B: compute and display ablation results (Tables 4 & 5)
# =========================

ablation_base = REPO_DIR / "logs" / "reproduction" / "ablation"

# Collect metrics for each ablation and the full (non-ablated) main results
all_ablation_rows = []

# Full model (no ablation) as reference
main_df = compute_cte_metrics(REPO_DIR / "logs" / "reproduction" / "main", MAIN_KEYS)
if not main_df.empty:
    for _, r in main_df.iterrows():
        all_ablation_rows.append(
            {
                "ablation": "full_model",
                "detector": r["detector"],
                "dataset": r["dataset"],
                "method": r["method"],
                "accuracy": r["accuracy"],
                "f1": r["f1"],
            }
        )

# Each ablation variant
for abl_name in ABLATIONS:
    abl_dir = ablation_base / abl_name
    if not abl_dir.exists():
        continue
    abl_df = compute_cte_metrics(abl_dir, MAIN_KEYS)
    for _, r in abl_df.iterrows():
        all_ablation_rows.append(
            {
                "ablation": abl_name,
                "detector": r["detector"],
                "dataset": r["dataset"],
                "method": r["method"],
                "accuracy": r["accuracy"],
                "f1": r["f1"],
            }
        )

abl_combined = pd.DataFrame(all_ablation_rows)

if abl_combined.empty:
    print("No ablation results found. Run Cell 9A first.")
else:
    # Focus on MoSEs methods for ablation comparison
    moses_only = abl_combined[abl_combined["method"].isin(["logistic", "xg"])]

    # Table 4: SAR ablation (H4) -- full_model vs no_router
    sar_abl = moses_only[moses_only["ablation"].isin(["full_model", "no_router"])]
    if not sar_abl.empty:
        print(f"\n{'='*80}")
        print("  Table 4: SAR Ablation (H4) -- MoSEs accuracy with vs without router")
        print(f"{'='*80}")
        sar_pivot = sar_abl.pivot_table(
            index=["detector", "method"],
            columns="ablation",
            values="accuracy",
            aggfunc="mean",
        )
        if "full_model" in sar_pivot.columns and "no_router" in sar_pivot.columns:
            sar_pivot["delta"] = sar_pivot["full_model"] - sar_pivot["no_router"]
        sar_pivot.columns.name = None
        print(sar_pivot.to_string(float_format=lambda x: f"{x:.4f}"))

    # Table 5: Conditional feature ablation (H3)
    cond_abl = moses_only[
        moses_only["ablation"].isin(
            ["full_model", "no_embedding", "no_condition", "no_pca"]
        )
    ]
    if not cond_abl.empty:
        print(f"\n{'='*80}")
        print("  Table 5: Conditional Feature Ablation (H3) -- MoSEs accuracy")
        print(f"{'='*80}")
        cond_pivot = cond_abl.pivot_table(
            index=["detector", "method"],
            columns="ablation",
            values="accuracy",
            aggfunc="mean",
        )
        if "full_model" in cond_pivot.columns:
            for col in ["no_embedding", "no_condition", "no_pca"]:
                if col in cond_pivot.columns:
                    cond_pivot[f"d_{col}"] = cond_pivot["full_model"] - cond_pivot[col]
        cond_pivot.columns.name = None
        print(cond_pivot.to_string(float_format=lambda x: f"{x:.4f}"))

    # Save ablation metrics
    abl_csv = RESULTS_ROOT / "ablation_metrics_detailed.csv"
    abl_combined.to_csv(abl_csv, index=False)
    print(f"\nAblation metrics saved to: {abl_csv}")

sync_outputs_to_drive()